# CMIP6 BCC-CSM2-MR monthly vas subset stalled job

When: 2026-10-31

The request subsets monthly northward near-surface wind (`vas`) from BCC-CSM2-MR, scenario `ssp245`, realization `r1i1p1f1`, version `v20190314`, for all months in 2092 over area `-1.965531,44.356247,0.582198,46.155833` (west, south, east, north).

The service reported no status update for at least 245 minutes. This does not establish the cause of the stall. The reproduction below preserves both supplied time filters and has not been run here.


## Reported WPS status

```xml
<wps:Status creationTime="2026-10-31T09:56:05Z">
    <wps:ProcessFailed>
        <wps:ExceptionReport>
            <ows:Exception exceptionCode="NoApplicableCode" locator="None">
                <ows:ExceptionText>Process failed: stalled-job recovery found no status update for at least 245 minutes.</ows:ExceptionText>
            </ows:Exception>
        </wps:ExceptionReport>
    </wps:ProcessFailed>
</wps:Status>
```


## Build the request

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` to test another deployment.


In [ ]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [ ]:
vas = ops.Input(
    "vas",
    ["c3s-cmip6.ScenarioMIP.BCC.BCC-CSM2-MR.ssp245.r1i1p1f1.Amon.vas.gn.v20190314"],
)
workflow = ops.Subset(
    vas,
    area="-1.965531,44.356247,0.582198,46.155833",
    time_components="month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:2092",
    time="2092/2092",
)

json.loads(workflow._serialise())


## Run and inspect the response

Running the next cell submits the workflow. It prints the status and successful output URLs without downloading data.


In [ ]:
response = workflow.orchestrate()

print(f"Succeeded: {response.ok}")
print(f"Status: {response.status}")
if response.ok:
    for url in response.download_urls():
        print(url)


## Captured log (Insights)

The supplied Insights summary labels this group `Timeout (3 failures)` and lists one request for this dataset and selection. The excerpt does not show how the three failures relate to that request.

```text
Timeout (3 failures)
    Dataset: c3s-cmip6.ScenarioMIP.BCC.BCC-CSM2-MR.ssp245.r1i1p1f1.Amon.vas.gn.v20190314
      1 request
        Selection: years=2092  time=2092/2092
        Reason: rved for non-replication superuser connections The above exception was the direct
                cause of the following exception: Traceback (most recent call last): File
                "/usr/local/Miniconda3-py312_24.9.2-0-Linux-x86_64/envs/rook/bin/joblauncher", line
                9, in <module> sys.exit(launcher()) ~~~~~~~~^^ File "/u [..]
        Job ID: 5cb971c4-d4e6-11f1-bef0-fa163e934c9b
        Recovered XML: /var/lib/pywps/job-incidents/rook/20261031T085605Z__recovered__rook__orchestrate__5cb971c4-d4e6-11f1-bef0-fa163e934c9b.xml
```

### Investigation notes

There was probably an issue with the database connection. The surviving phrase `rved for non-replication superuser connections` suggests PostgreSQL connection-slot exhaustion. The full traceback is no longer available, so the exact failure cannot be confirmed.

The recovered WPS status records a prolonged absence of status updates, which may have resulted from that database connection issue. The excerpt does not establish that the subset computation itself timed out. The recovered XML path is recorded as supplied; its contents have not been inspected here.
